# 02 — Camada Silver | CineData Analytics

**Objetivo:** limpar, tipar, traduzir e deduplicar os dados da Bronze. 

**Princípios aplicados em todas as tabelas**
1. **Colunas em português** e tipagem correta.
2. **Conversão segura de tipos**: uso de `try_cast` e uma função de limpeza numérica própria, que **valida o texto inteiro** antes de converter.
3. **Deduplicação pela ingestão mais recente** (`ingestion_datetime`), com desempate pela linha mais completa.
4. A ordem das seções respeita as dependências: a **cotação do dólar vem antes do financeiro** (que a utiliza).

## 1. Configuração e funções utilitárias

In [ ]:
from datetime import date
from functools import reduce

from pyspark.sql import functions as F, Window, DataFrame

CATALOG = "workspace"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

def salvar_silver(df: DataFrame, tabela: str) -> None:
    """Silver é reprocessável: sobrescreve a tabela inteira a partir da Bronze."""
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela))
    print(f"{tabela}: {spark.table(tabela).count()} linhas")

In [ ]:
# Funções utilitárias reutilizadas nas tabelas Silver

def deduplicar_mais_recente(df: DataFrame, chave: list) -> DataFrame:
    """
    Mantém 1 linha por chave: a de maior ingestion_datetime.
    Empate (mesma carga) -> fica a linha com mais colunas preenchidas, para não perder informação.
    """
    completude = sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in df.columns)
    w = Window.partitionBy(*chave).orderBy(F.col("ingestion_datetime").desc(), F.col("_completude").desc())
    return (
        df.withColumn("_completude", completude)
          .withColumn("_rn", F.row_number().over(w))
          .filter("_rn = 1")
          .drop("_rn", "_completude")
    )


def texto_ou_nulo(c):
    """trim + string vazia vira NULL."""
    t = F.trim(c.cast("string"))
    return F.when(t != "", t)


def normalizar_id(c):
    """Chave natural como STRING: remove espaços e o sufixo '.0' (ex.: 862.0 -> 862)."""
    t = F.regexp_replace(F.trim(c.cast("string")), r"\.0+$", "")
    return F.when(t != "", t)


def numero_limpo(c, milhar_ambiguo: bool = True):
    """
    Converte texto numérico 'sujo' em texto numérico canônico (ponto decimal) ou NULL.
    Só aceita se o texto INTEIRO tiver formato numérico -> textos deslocados viram NULL.

    Formatos tratados: 1234 | 1234.56 | 1234,56 | 1.234,56 | 1,234.56 | 1.234.567 | 1,234,567 | -5

    milhar_ambiguo controla o caso de UM único separador seguido de 3 dígitos ('1.234' / '1,234'):
      True  -> é separador de milhar (dinheiro e contagens: 1.234 = 1234)
      False -> é separador decimal  (popularidade/notas: 1,234 = 1.234)
    """
    s = F.regexp_replace(F.trim(c.cast("string")), r"\s", "")
    q = "+" if milhar_ambiguo else "{2,}"          # nº mínimo de grupos de 3 dígitos
    br_dec       = s.rlike(r"^-?\d{1,3}(\.\d{3})+,\d+$")               # 1.234,56
    us_dec       = s.rlike(r"^-?\d{1,3}(,\d{3})+\.\d+$")               # 1,234.56
    milhar_ponto = s.rlike(rf"^-?\d{{1,3}}(\.\d{{3}}){q}$")              # 1.234  /  1.234.567
    milhar_virg  = s.rlike(rf"^-?\d{{1,3}}(,\d{{3}}){q}$")               # 1,234  /  1,234,567
    virg_decimal = s.rlike(r"^-?\d*,\d+$")                               # 12,5
    simples      = s.rlike(r"^-?(\d+\.?\d*|\.\d+)$")                  # 12 / 12.5 / .5
    return (
        F.when(br_dec,       F.regexp_replace(F.regexp_replace(s, r"\.", ""), ",", "."))
         .when(us_dec,       F.regexp_replace(s, ",", ""))
         .when(milhar_ponto, F.regexp_replace(s, r"\.", ""))
         .when(milhar_virg,  F.regexp_replace(s, ",", ""))
         .when(virg_decimal, F.regexp_replace(s, ",", "."))
         .when(simples,      s)
    )   # sem .otherwise -> NULL


def add_decimal(df, saida, c, tipo="DOUBLE", milhar_ambiguo=True):
    """Adiciona coluna numérica com try_cast (nunca interrompe o pipeline)."""
    return (df.withColumn("_n", numero_limpo(c, milhar_ambiguo))
              .withColumn(saida, F.expr(f"try_cast(_n AS {tipo})"))
              .drop("_n"))


def add_inteiro(df, saida, c):
    """Inteiro seguro: aceita só valores sem parte fracionária e dentro do range de INT."""
    df = add_decimal(df, "_d", c, "DOUBLE", True)
    return (df.withColumn(saida, F.when((F.col("_d") == F.floor("_d")) & F.col("_d").between(-2147483648, 2147483647),
                                        F.col("_d").cast("int")))
              .drop("_d"))


def add_dinheiro(df, saida, c):
    """
    Higieniza valores monetários: remove símbolos (US$, R$, $, €, £), siglas, espaços e separadores de milhar;
    entende sufixos K/M/B (ex.: '1.5M' = 1.500.000); marcadores de ausência ('Unknown', 'Não Informado', 'N/A'...)
    não têm dígitos e por isso viram NULL. Zero e negativos também viram NULL (regra de negócio).
    """
    bruto = F.upper(F.trim(c.cast("string")))
    sem_moeda = F.regexp_replace(bruto, r"(US\$|R\$|USD|BRL|\$|€|£|\s)", "")
    mult = (F.when(sem_moeda.rlike(r"[0-9]K$"), 1_000)
             .when(sem_moeda.rlike(r"[0-9]M$"), 1_000_000)
             .when(sem_moeda.rlike(r"[0-9]B$"), 1_000_000_000)
             .otherwise(1))
    base = F.regexp_replace(sem_moeda, r"[KMB]$", "")
    df = (df.withColumn("_base", base)
            .withColumn("_mult", mult.cast("decimal(20,0)"))
            .withColumn("_num", numero_limpo(F.col("_base"), True))
            .withColumn(saida, F.expr("try_cast(try_cast(_num AS DECIMAL(30,4)) * _mult AS DECIMAL(18,2))"))
            .drop("_base", "_mult", "_num"))
    return df.withColumn(saida, F.when(F.col(saida) > 0, F.col(saida)))   # <=0 -> ausente


# Formatos de data observados/possíveis na origem. A ORDEM importa para datas ambíguas (05/06/2020):
# assumimos padrão brasileiro (dd/MM) antes do americano (MM/dd). Datas com dia > 12 são inequívocas.
FORMATOS_DATA = [
    "yyyy-M-d", "yyyy/M/d",
    "d/M/yyyy", "d-M-yyyy", "d.M.yyyy",
    "M/d/yyyy", "M-d-yyyy",
    "yyyyMMdd",
    "MMM d, yyyy", "MMMM d, yyyy", "d MMM yyyy", "d MMMM yyyy",
]

def add_data(df, saida, c):
    """Tenta cada formato em sequência (coalesce); só vira NULL se NENHUM formato servir."""
    txt = F.trim(c.cast("string"))
    iso = F.regexp_extract(txt, r"^(\d{4}-\d{2}-\d{2})[T ].*$", 1)       # '2020-01-05 00:00:00' -> '2020-01-05'
    df = df.withColumn("_txt", F.when(iso != "", iso).otherwise(txt))
    tentativas = [F.expr(f"CAST(try_to_timestamp(_txt, '{f}') AS DATE)") for f in FORMATOS_DATA]
    return df.withColumn(saida, F.coalesce(*tentativas)).drop("_txt")

## 2. `silver.tb_cotacao_dolar`

Verificação de 1 cotação por dia útil: como a Bronze é `append` (rodar o notebook 01 mais de uma vez repete linhas), mantemos **1 registro por dia** com a ingestão mais recente.
Criamos um **calendário contínuo** (todos os dias, inclusive fins de semana/feriados) e aplicamos **Forward Fill**: dia sem cotação herda a do último dia útil anterior. O calendário vai até hoje, para que sábados/domingos recentes também sejam preenchidos.

In [ ]:
cot = (
    spark.table("bronze.tb_cotacao_dolar")
    .withColumn("data_cotacao", F.expr("CAST(try_to_timestamp(substring(dataHoraCotacao, 1, 10), 'yyyy-MM-dd') AS DATE)"))
    .filter(F.col("data_cotacao").isNotNull() & F.col("cotacaoCompra").isNotNull() & (F.col("cotacaoCompra") > 0))
)

w_dia = Window.partitionBy("data_cotacao").orderBy(F.col("dataHoraCotacao").desc(), F.col("ingestion_datetime").desc())
cot_dia = (
    cot.withColumn("_rn", F.row_number().over(w_dia)).filter("_rn = 1")
       .select("data_cotacao", F.col("cotacaoCompra").cast("decimal(18,6)").alias("cotacao_dolar_brl"))
)

lim = cot_dia.agg(F.min("data_cotacao").alias("ini"), F.max("data_cotacao").alias("fim")).first()
assert lim["ini"] is not None, "Bronze sem cotações válidas — rode o notebook 01 (seção da API) antes."
fim_calendario = max(lim["fim"], date.today())

calendario = spark.range(1).select(F.explode(F.sequence(F.lit(lim["ini"]), F.lit(fim_calendario))).alias("data_cotacao"))

w_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)
silver_cotacao = (
    calendario.join(cot_dia.withColumnRenamed("cotacao_dolar_brl", "_cot"), "data_cotacao", "left")
    .withColumn("cotacao_dolar_brl", F.last("_cot", ignorenulls=True).over(w_ffill))   # Forward Fill
    .withColumn("cotacao_preenchida", F.col("_cot").isNull())                          # True = valor herdado
    .select("data_cotacao", "cotacao_dolar_brl", "cotacao_preenchida")
    .orderBy("data_cotacao")
)

salvar_silver(silver_cotacao, "silver.tb_cotacao_dolar")
display(spark.table("silver.tb_cotacao_dolar").orderBy(F.col("data_cotacao").desc()))

## 3. `silver.tb_info_filmes`

| Regra | Implementação |
|---|---|
| Unicidade por filme | `row_number` por `id_filme` ordenando pela ingestão mais recente |
| Status | normaliza (remove hífens/ruídos/caixa) → traduz → o que não mapear vira **"Não Informado"** |
| Data multi-formato | `add_data` testa vários padrões em sequência |
| `ano_lancamento` | `year(data_lancamento)` |
| `duracao_minutos` | aceita `142`, `142 min`, `2h 22m`; zero/negativo → NULL |

In [ ]:
MAPA_STATUS = {          # chave = status normalizado (só letras minúsculas, sem espaços/hífens)
    "released":       "Lançado",
    "postproduction": "Pós-Produção",
    "inproduction":   "Em Produção",
    "planned":        "Planejado",
    "rumored":        "Rumores",
    "rumoured":       "Rumores",
    "canceled":       "Cancelado",
    "cancelled":      "Cancelado",
}

# 'Post-Production', 'post_production', ' POST PRODUCTION ' -> 'postproduction'
chave_status = F.lower(F.regexp_replace(F.trim(F.col("status")), r"[^A-Za-z]", ""))
status_traduzido = F.lit("Não Informado")                      # corrompido / não mapeável
for chave, traducao in MAPA_STATUS.items():
    status_traduzido = F.when(chave_status == chave, traducao).otherwise(status_traduzido)

info = (
    spark.table("bronze.tb_movies_info")
    .withColumn("id_filme", normalizar_id(F.col("id")))
    .filter(F.col("id_filme").isNotNull())
)
info = deduplicar_mais_recente(info, ["id_filme"])

# duração: '2h 22m' | '142 min' | '142.0' | '142'
info = (
    info
    .withColumn("_rt", F.lower(F.trim(F.col("runtime"))))
    .withColumn("_h", F.regexp_extract("_rt", r"^(\d+)\s*h", 1))
    .withColumn("_m", F.regexp_extract("_rt", r"^\d+\s*h\w*\s*(\d+)\s*m", 1))
    .withColumn("_plain", numero_limpo(F.regexp_replace("_rt", r"\s*(minutos|minutes|min|m)$", ""), milhar_ambiguo=False))
    .withColumn("duracao_minutos", F.expr("""
        CASE WHEN _h <> '' THEN try_cast(_h AS INT) * 60 + coalesce(try_cast(nullif(_m, '') AS INT), 0)
             ELSE try_cast(floor(try_cast(_plain AS DOUBLE)) AS INT) END
    """))
    .withColumn("duracao_minutos", F.when(F.col("duracao_minutos") > 0, F.col("duracao_minutos")))
)

info = add_data(info, "data_lancamento", F.col("release_date"))

silver_info = info.select(
    "id_filme",
    texto_ou_nulo(F.col("title")).alias("titulo"),
    texto_ou_nulo(F.col("original_title")).alias("titulo_original"),
    "data_lancamento",
    F.year("data_lancamento").alias("ano_lancamento"),
    "duracao_minutos",
    F.lower(texto_ou_nulo(F.col("original_language"))).alias("idioma_original"),
    status_traduzido.alias("status_filme"),
    texto_ou_nulo(F.col("overview")).alias("sinopse"),
    texto_ou_nulo(F.col("tagline")).alias("frase_divulgacao"),
)

salvar_silver(silver_info, "silver.tb_info_filmes")
display(spark.table("silver.tb_info_filmes").limit(10))

In [ ]:
# Diagnóstico: distribuição do status e datas que não puderam ser convertidas
display(spark.table("silver.tb_info_filmes").groupBy("status_filme").count().orderBy(F.desc("count")))

nao_convertidas = (
    spark.table("bronze.tb_movies_info").select(F.col("release_date").alias("release_date_bronze"))
    .distinct()
    .transform(lambda d: add_data(d, "convertida", F.col("release_date_bronze")))
    .filter(F.col("release_date_bronze").isNotNull() & (F.trim("release_date_bronze") != "") & F.col("convertida").isNull())
)
print("Valores de data não convertíveis (viram NULL):", nao_convertidas.count())
display(nao_convertidas.limit(20))

## 4. `silver.tb_financeiro_filmes`

- Higienização de `budget`/`revenue` (símbolos, milhar, sufixos K/M/B, "Unknown"/"Não Informado" → NULL); **zero e negativos → NULL**.
- **BRL** = USD × cotação do dólar **mais recente** da `silver.tb_cotacao_dolar` (taxa única, conforme enunciado: "taxa de cotação obtida").
- **Lucro** = receita − orçamento, com ausentes tratados como 0 na subtração (o enunciado pede que valores ausentes *não invalidem* o resultado). Se **ambos** forem ausentes, o lucro continua NULL.
- **Margem %** = lucro ÷ receita × 100, com `try_divide` (receita zero/NULL → NULL, sem erro de divisão por zero).

In [ ]:
AUSENTE_COMO_ZERO_NO_LUCRO = True

fin = (
    spark.table("bronze.tb_movies_financials")
    .withColumn("id_filme", normalizar_id(F.col("id")))
    .filter(F.col("id_filme").isNotNull())
)
fin = deduplicar_mais_recente(fin, ["id_filme"])

fin = add_dinheiro(fin, "orcamento_usd", F.col("budget"))
fin = add_dinheiro(fin, "receita_usd",   F.col("revenue"))

# Cotação mais recente da série contínua
linha_cot = spark.table("silver.tb_cotacao_dolar").orderBy(F.col("data_cotacao").desc()).first()
taxa = linha_cot["cotacao_dolar_brl"]
print(f"Cotação utilizada: R$ {taxa} (referência {linha_cot['data_cotacao']})")

orc, rec = F.col("orcamento_usd"), F.col("receita_usd")
if AUSENTE_COMO_ZERO_NO_LUCRO:
    lucro_usd = F.when(orc.isNotNull() | rec.isNotNull(), F.coalesce(rec, F.lit(0)) - F.coalesce(orc, F.lit(0)))
else:
    lucro_usd = F.when(orc.isNotNull() & rec.isNotNull(), rec - orc)

silver_fin = (
    fin
    .withColumn("orcamento_brl", F.expr(f"try_cast(orcamento_usd * {taxa} AS DECIMAL(18,2))"))
    .withColumn("receita_brl",   F.expr(f"try_cast(receita_usd * {taxa} AS DECIMAL(18,2))"))
    .withColumn("lucro_usd", lucro_usd.cast("decimal(18,2)"))
    .withColumn("lucro_brl", F.expr(f"try_cast(lucro_usd * {taxa} AS DECIMAL(18,2))"))
    .withColumn("margem_lucro_pct", F.expr("try_cast(round(try_divide(lucro_usd, receita_usd) * 100, 2) AS DECIMAL(18,2))"))
    .select("id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
            "lucro_usd", "lucro_brl", "margem_lucro_pct")
)

salvar_silver(silver_fin, "silver.tb_financeiro_filmes")
display(spark.table("silver.tb_financeiro_filmes").limit(20))

## 5. `silver.tb_metricas_engajamento`

- `popularidade` e notas: separador decimal ambíguo → `numero_limpo(..., milhar_ambiguo=False)` (`"1,234"` = 1.234; `"1.234,5"` = 1234.5).
- Contagens de votos: separador de milhar removido; só inteiros válidos.
- **Column shift**: qualquer texto que não seja número por inteiro vira `NULL` (sem quebrar o pipeline).
- **Regras de negócio**: notas fora de `[0, 10]` (inclusive escala errada, ex.: `85`) → NULL; popularidade/votos negativos → NULL.

In [ ]:
met = (
    spark.table("bronze.tb_movies_metrics")
    .withColumn("id_filme", normalizar_id(F.col("id")))
    .filter(F.col("id_filme").isNotNull())
)
met = deduplicar_mais_recente(met, ["id_filme"])

met = add_decimal(met, "popularidade",    F.col("popularity"),    "DOUBLE", milhar_ambiguo=False)
met = add_decimal(met, "nota_media_tmdb", F.col("vote_average"),  "DOUBLE", milhar_ambiguo=False)
met = add_decimal(met, "nota_media_imdb", F.col("averageRating"), "DOUBLE", milhar_ambiguo=False)
met = add_inteiro(met, "qtd_votos_tmdb",  F.col("vote_count"))
met = add_inteiro(met, "qtd_votos_imdb",  F.col("numVotes"))

def nota_valida(c): return F.when(c.between(0, 10), c)      # fora de 0..10 -> NULL
def nao_negativo(c): return F.when(c >= 0, c)               # negativo -> NULL

silver_met = met.select(
    "id_filme",
    nao_negativo(F.col("popularidade")).alias("popularidade"),
    nota_valida(F.col("nota_media_tmdb")).alias("nota_media_tmdb"),
    nao_negativo(F.col("qtd_votos_tmdb")).alias("qtd_votos_tmdb"),
    nota_valida(F.col("nota_media_imdb")).alias("nota_media_imdb"),
    nao_negativo(F.col("qtd_votos_imdb")).alias("qtd_votos_imdb"),
)

salvar_silver(silver_met, "silver.tb_metricas_engajamento")
display(spark.table("silver.tb_metricas_engajamento").limit(20))

## 6. `silver.tb_avaliacoes_usuarios`

- Nota: conversão segura + faixa **0–10** (fora → NULL).
- Comentário vazio/só espaços/NULL → `"Sem comentário"`.
- Unicidade: duplicatas **integrais da origem** (filme + usuário + nota + comentário idênticos, como vieram) são removidas **antes** da limpeza.

In [ ]:
# 1) Duplicatas integrais da ORIGEM (mesmo filme + usuário + nota + comentário, como vieram) são removidas ANTES da limpeza.
rev = spark.table("bronze.tb_movies_reviews").dropDuplicates(["id", "nome", "nota", "comentario"])
rev = rev.withColumn("id_filme", normalizar_id(F.col("id"))).filter(F.col("id_filme").isNotNull())
rev = add_decimal(rev, "nota_usuario", F.col("nota"), "DOUBLE", milhar_ambiguo=False)

silver_rev = (
    rev.select(
        "id_filme",
        texto_ou_nulo(F.col("nome")).alias("nome_usuario"),
        F.when(F.col("nota_usuario").between(0, 10), F.col("nota_usuario")).alias("nota_usuario"),
        F.coalesce(texto_ou_nulo(F.col("comentario")), F.lit("Sem comentário")).alias("comentario_usuario"),
    )
)

salvar_silver(silver_rev, "silver.tb_avaliacoes_usuarios")
display(spark.table("silver.tb_avaliacoes_usuarios").limit(20))

## 7. `credits_and_tags` → `silver.tb_generos` e `silver.tb_pessoas_empresas`

Primeiro mantem **uma única versão por filme** (ingestão mais recente). Depois:

**Gêneros** — padroniza separadores (`,` `;` `|` `/`), remove colchetes/aspas, faz `split + explode` e **valida contra o domínio oficial de gêneros do TMDB** (qualquer coisa fora dele — números, descrições, dados deslocados — é descartada). A comparação ignora caixa, espaços e hífens (`"sci-fi"`/`"Science  Fiction"` → `Science Fiction`).

**Pessoas e empresas** — as 4 colunas (`cast`, `directors`, `writers`, `production_companies`) são explodidas e unificadas com o `tipo_entidade`; capitalização padronizada (`initcap`); ruídos descartados (vazios, números, URLs, placeholders, nomes de gêneros/status que "vazaram" para a coluna, textos muito longos); duplicados removidos.

In [ ]:
cred = spark.table("bronze.tb_credits_and_tags").withColumn("id_filme", normalizar_id(F.col("id")))
cred = deduplicar_mais_recente(cred.filter(F.col("id_filme").isNotNull()), ["id_filme"])

# Domínio oficial de gêneros do TMDB. Chave = só letras minúsculas.
GENEROS_TMDB = ["Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama", "Family",
                "Fantasy", "History", "Horror", "Music", "Mystery", "Romance", "Science Fiction",
                "TV Movie", "Thriller", "War", "Western"]
MAPA_GENEROS = {g.lower().replace(" ", ""): g for g in GENEROS_TMDB}
MAPA_GENEROS["scifi"] = "Science Fiction"          # variação comum

def chave_letras(c):
    return F.lower(F.regexp_replace(F.trim(c), r"[^A-Za-z]", ""))

genero_canonico = F.lit(None).cast("string")
for chave, nome in MAPA_GENEROS.items():
    genero_canonico = F.when(chave_letras(F.col("genero_bruto")) == chave, nome).otherwise(genero_canonico)

silver_generos = (
    cred.select("id_filme", F.col("genres"))
    .withColumn("lista", F.regexp_replace("genres", r'[\[\]"\']', ""))        # remove [ ] " '
    .withColumn("lista", F.regexp_replace("lista", r"[;|/]", ","))              # unifica separadores
    .withColumn("genero_bruto", F.explode(F.split("lista", ",")))               # split + explode
    .withColumn("genero", genero_canonico)
    .filter(F.col("genero").isNotNull())                                        # descarta resíduos/deslocados
    .select("id_filme", "genero")
    .dropDuplicates()
)

salvar_silver(silver_generos, "silver.tb_generos")
display(spark.table("silver.tb_generos").groupBy("genero").count().orderBy(F.desc("count")))

In [ ]:
TIPOS_ENTIDADE = {"cast": "Ator", "directors": "Diretor", "writers": "Roteirista", "production_companies": "Produtora"}

# Valores que indicam ausência ou que pertencem a outros domínios (vazamento por column shift)
PLACEHOLDERS = {"n/a", "na", "nan", "null", "none", "unknown", "desconhecido", "[]", "nao informado", "não informado"}
DOMINIO_OUTROS = set(MAPA_GENEROS.keys()) | set(MAPA_STATUS.keys())     # chaves 'só letras'

def explodir_entidades(coluna: str, tipo: str) -> DataFrame:
    return (
        cred.select("id_filme", F.col(coluna).alias("bruto"))
        .withColumn("lista", F.regexp_replace("bruto", r'[\[\]"]', ""))
        .withColumn("lista", F.regexp_replace("lista", r"[;|]", ","))
        .withColumn("nome", F.explode(F.split("lista", ",")))
        .withColumn("nome", F.regexp_replace(F.trim("nome"), r"^'+|'+$", ""))       # aspas simples nas bordas
        .withColumn("nome", F.regexp_replace("nome", r"\s+", " "))                  # espaços repetidos
        .withColumn("tipo_entidade", F.lit(tipo))
        .select("id_filme", "nome", "tipo_entidade")
    )

pe = reduce(DataFrame.unionByName, [explodir_entidades(c, t) for c, t in TIPOS_ENTIDADE.items()])

nome_lower = F.lower(F.trim("nome"))
silver_pe = (
    pe
    .filter(F.col("nome").isNotNull() & (F.length("nome") >= 2) & (F.length("nome") <= 80))   # vazio / texto longo (descrição deslocada)
    .filter(F.col("nome").rlike(r"[A-Za-zÀ-ÿ]"))                                           # exige ao menos uma letra (descarta números/datas)
    .filter(~F.col("nome").rlike(r"(?i)^(https?://|www\.)"))                              # URLs
    .filter(~nome_lower.isin(*PLACEHOLDERS))
    .filter(~chave_letras(F.col("nome")).isin(*DOMINIO_OUTROS))                            # 'Drama', 'Released' dentro de elenco/produtora
    .filter(F.size(F.split("nome", " ")) <= 8)                                             # frase longa = descrição deslocada
    .withColumn("nome", F.initcap(F.trim("nome")))                                         # padroniza capitalização
    .dropDuplicates(["id_filme", "nome", "tipo_entidade"])
)

salvar_silver(silver_pe, "silver.tb_pessoas_empresas")
display(spark.table("silver.tb_pessoas_empresas").groupBy("tipo_entidade").count())

In [ ]:
# Diagnóstico (opcional): inspecione os nomes mais longos para ajustar os filtros de ruído, se necessário
display(
    spark.table("silver.tb_pessoas_empresas")
    .select("nome", "tipo_entidade", F.length("nome").alias("tam"))
    .distinct().orderBy(F.desc("tam")).limit(30)
)

## 8. Validação final da Silver

In [ ]:
def checar_unicidade(tabela, chave):
    df = spark.table(tabela)
    total, distintos = df.count(), df.select(*chave).distinct().count()
    print(f"{'✔' if total == distintos else '✘'} {tabela:<38} chave={chave}  linhas={total}  distintas={distintos}")

checar_unicidade("silver.tb_info_filmes",           ["id_filme"])
checar_unicidade("silver.tb_financeiro_filmes",     ["id_filme"])
checar_unicidade("silver.tb_metricas_engajamento",  ["id_filme"])
# Reviews: a dedup é sobre a linha ORIGINAL (antes da limpeza). Pode sobrar par idêntico só após a limpeza
# (ex.: nota '9' e '9.0') — isso é informativo, não erro.
_rv = spark.table("silver.tb_avaliacoes_usuarios")
print(f"ℹ silver.tb_avaliacoes_usuarios  linhas={_rv.count()}  iguais_apos_limpeza={_rv.count() - _rv.dropDuplicates().count()}")
checar_unicidade("silver.tb_generos",               ["id_filme", "genero"])
checar_unicidade("silver.tb_pessoas_empresas",      ["id_filme", "nome", "tipo_entidade"])
checar_unicidade("silver.tb_cotacao_dolar",         ["data_cotacao"])

for t in ["silver.tb_info_filmes", "silver.tb_financeiro_filmes", "silver.tb_metricas_engajamento"]:
    print(f"\n{t}"); spark.table(t).printSchema()